# Fake News Detection - Claim Retrieval

## Student: Santiago Lares Harbin

## Tutor: Juan Manuel Pérez

This notebook is part of the Fake News Detection project. It is the first step of the project, where we will generate embeddings from the social posts and the checked facts. Then, we will use these embeddings to retrieve the most similar checked facts for each social post. This way, we will be able to use the fact-checking information to help us classify the social posts. This project will be presented on SemEval 2025 Shared Task 7 Competition: https://disai.eu/semeval-2025/.

# Basic Retrieval System

## Load Data

In [ ]:
import pandas as pd

from utils.load import LoadDataCSV

# Load datasets

df_loader=LoadDataCSV()

In [51]:

df_fact_checks, df_posts, df_fact_check_post_mapping = df_loader.load_data()

In [ ]:
import pandas as pd

from utils.load import LoadDataCSV

# Load datasets

df_loader=LoadDataCSV()


monolingual_data= df_loader.split_data()

In [7]:
fact_checks_mono = monolingual_data["fact_checks"]
posts_train_mono = monolingual_data["posts_train"]
posts_dev_mono = monolingual_data["posts_dev"]

In [8]:
# Corrected logic for combining OCR and text into ocr_text column
def combine_ocr_text(row):
    ocr = row['ocr']
    text = row['text']
    if ocr and isinstance(ocr, list) and len(ocr) > 0 and len(ocr[0]) > 0:
        return 'passage: '+ str(ocr[0][0]) + ' ' + (str(text[0]) if text else '')
    elif text:
        return 'passage: '+ str(text[0])
    else:
        return ''

# Apply the function to create the new column
posts_train_mono['ocr_text'] = posts_train_mono.apply(combine_ocr_text, axis=1)
posts_dev_mono['ocr_text'] = posts_dev_mono.apply(combine_ocr_text, axis=1)

In [9]:
# Corrected logic for combining OCR and text into ocr_text column
def combine_ocr_text_english(row):
    ocr = row['ocr']
    text = row['text']
    if ocr and isinstance(ocr, list) and len(ocr) > 0 and len(ocr[0]) > 0:
        return 'passage: '+str(ocr[0][1]) + ' ' + (str(text[1]) if text else '')
    elif text:
        return 'passage: '+str(text[1])
    else:
        return ''

# Apply the function to create the new column
posts_train_mono['ocr_text_eng'] = posts_train_mono.apply(combine_ocr_text_english, axis=1)
posts_dev_mono['ocr_text_eng'] = posts_dev_mono.apply(combine_ocr_text_english, axis=1)

In [10]:
# Corrected logic for combining OCR and text into ocr_text column
def combine_claim_text(row):
    claim = row['claim']
    title = row['title']
    if claim and isinstance(claim, list) and len(claim) > 0 and len(claim[0]) > 0:
        return 'query: '+ str(claim[0][0]) + ' ' + (str(title[0]) if title else '')
    elif title:
        return 'query: '+ str(title[0])
    else:
        return ''

# Apply the function to create the new column
fact_checks_mono['claim_title'] = fact_checks_mono.apply(combine_claim_text, axis=1)

In [11]:
# Corrected logic for combining OCR and text into ocr_text column
def combine_claim_text_eng(row):
    claim = row['claim']
    title = row['title']
    if claim and isinstance(claim, list) and len(claim) > 0 and len(claim[0]) > 0:
        return 'query: '+ str(claim[0][1]) + ' ' + (str(title[1]) if title else '')
    elif title:
        return 'query: '+ str(title[1])
    else:
        return ''

# Apply the function to create the new column
fact_checks_mono['claim_title_eng'] = fact_checks_mono.apply(combine_claim_text_eng, axis=1)

In [184]:
def prepare_df_for_saving(df, embedding_columns):
    # Create a copy to avoid modifying original
    df_to_save = df.copy()

    for col in embedding_columns:
    
        # Convert numpy arrays to list format
        df_to_save[col] = df_to_save[col].apply(
            lambda x: x.tolist() if isinstance(x, np.ndarray) else x
        )
    
    return df_to_save

# Embeddings generation

In [12]:
from sentence_transformers import SentenceTransformer, util
import torch

/Users/santiagolares/Projects/semeval_2025/tformervenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# Load the model to create embeddings
small_model = SentenceTransformer("intfloat/multilingual-e5-small", trust_remote_code=True)

In [14]:
small_model.to('mps')  # Mueve el modelo a MPS

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [15]:
import chromadb

# Configuration of ChromaDB and SentenceTransformer
client = chromadb.Client()

In [ ]:
posts_train_mono_sample=posts_train_mono.sample(100)
posts_dev_mono_sample=posts_dev_mono.sample(100)

In [ ]:
# Filtrar los pares en pairs que correspondan a los post_id de la muestra
pairs_sample = df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_train_mono_sample['post_id'])]

# Filtrar los fact-checks que correspondan a los fact_check_id en la muestra de pares
fact_checks_mono_sample_100 = fact_checks_mono[fact_checks_mono['fact_check_id'].isin(pairs_sample['fact_check_id'])]

In [90]:
posts_train_mono_sample_1000=posts_train_mono.sample(1000)

# Filtrar los pares en pairs que correspondan a los post_id de la muestra
pairs_sample_1000 = df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_train_mono_sample_1000['post_id'])]

# Filtrar los fact-checks que correspondan a los fact_check_id en la muestra de pares
fact_checks_mono_sample_1000 = fact_checks_mono[fact_checks_mono['fact_check_id'].isin(pairs_sample_1000['fact_check_id'])]

In [133]:
posts_train_mono_sample_1000.to_csv('posts_train_mono_sample_1000.csv', index=False)

fact_checks_mono_sample_1000.to_csv('fact_checks_mono_sample_1000.csv', index=False)

# Embeddings generation by batch

## Fact Checks

In [95]:
import torch

def embed_facts(df, model, column_name):
    batch_size = 32  # Ajusta este valor según la memoria disponible
    embeddings = []

    for i in range(0, len(df), batch_size):
        batch = df[column_name] [i:i + batch_size].tolist()
        batch_embeddings = model.encode(batch, convert_to_tensor=True, device='mps').cpu().numpy()
        embeddings.extend(batch_embeddings)
        torch.mps.empty_cache()  # Libera memoria en MPS (no siempre necesario pero puede ayudar)

    df[column_name+'_embedding'] = embeddings.copy()

    return df

fact_checks_mono_sample_1000 = embed_facts(fact_checks_mono_sample_1000, small_model, 'claim_title_eng')
fact_checks_mono_sample_1000=embed_facts(fact_checks_mono_sample_1000, small_model, 'claim_title')


/var/folders/dj/58tclkkj76sf6w2h7x_9cyt40000gn/T/ipykernel_1307/2219507543.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column_name+'_embedding'] = embeddings.copy()
/var/folders/dj/58tclkkj76sf6w2h7x_9cyt40000gn/T/ipykernel_1307/2219507543.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column_name+'_embedding'] = embeddings.copy()


In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small", trust_remote_code=True)

## Posts

In [21]:
import pandas as pd
import numpy as np
import torch

In [22]:

def split_into_chunks(text, max_tokens=512):
    """
    Divide un texto en chunks de hasta max_tokens tokens.
    """
    if not text:
        return []
    tokens = tokenizer.encode(text, truncation=True)
    chunks = [tokens[i:i + max_tokens] for i in range(0, len(tokens), max_tokens)]
    return [tokenizer.decode(chunk, skip_special_tokens=True) for chunk in chunks]

# Procesar DataFrame para agregar chunks
def add_chunks_column(df, text_column, max_tokens=512):
    df['chunks'] = df[text_column].apply(lambda x: split_into_chunks(x, max_tokens))
    return df

# Generar embeddings por chunks
def generate_chunk_embeddings_batched(df, embedding_model, new_column_name, batch_size=32, device="mps", max_tokens=512):
    """
    Genera embeddings para los chunks en batches para evitar problemas de memoria.
    Devuelve un DataFrame con una nueva columna 'chunk_embeddings'.
    """
    chunk_embeddings = []

    for index, row in df.iterrows():
        chunks = row['chunks']
        row_embeddings = []

                # Validar truncación
        valid_chunks = [
            tokenizer.decode(
                tokenizer.encode(chunk, truncation=True, max_length=max_tokens),
                skip_special_tokens=True
            )
            for chunk in chunks
        ]
        
        for i in range(0, len(valid_chunks), batch_size):
            batch = valid_chunks[i:i + batch_size]
            batch_embeddings = (
                embedding_model
                .encode(batch, convert_to_tensor=True, device=device)
                .cpu()  # Liberar memoria en GPU
                .numpy()
            )
            row_embeddings.extend(batch_embeddings)
            torch.mps.empty_cache()  # Libera memoria en MPS
        
        chunk_embeddings.append(row_embeddings)
    
    df[new_column_name] = chunk_embeddings
    return df

# Paso 1: Dividir en chunks
df_posts_chunked = add_chunks_column(posts_train_mono_sample, 'ocr_text', max_tokens=512)

# Paso 2: Generar embeddings por chunks
posts_train_mono_sample = generate_chunk_embeddings_batched(df_posts_chunked, small_model, new_column_name='ocr_text_embedding',device="mps")

# Paso 1: Dividir en chunks
df_posts_chunked = add_chunks_column(posts_train_mono_sample, 'ocr_text_eng', max_tokens=512)

# Paso 2: Generar embeddings por chunks
posts_train_mono_sample = generate_chunk_embeddings_batched(df_posts_chunked, small_model, new_column_name='ocr_text_eng_embedding',device="mps")

posts_train_mono_sample.to_csv('posts_train_mono_sample.csv', index=False)

# Paso 1: Dividir en chunks
df_posts_chunked = add_chunks_column(posts_dev_mono_sample, 'ocr_text', max_tokens=512)

# Paso 2: Generar embeddings por chunks
posts_dev_mono_sample = generate_chunk_embeddings_batched(df_posts_chunked, small_model, new_column_name='ocr_text_embedding',device="mps")

# Paso 1: Dividir en chunks
df_posts_chunked = add_chunks_column(posts_dev_mono_sample, 'ocr_text_eng', max_tokens=512)

# Paso 2: Generar embeddings por chunks
posts_dev_mono_sample = generate_chunk_embeddings_batched(df_posts_chunked, small_model, new_column_name='ocr_text_eng_embedding',device="mps")

posts_dev_mono_sample.to_csv('posts_dev_mono_sample.csv', index=False)

In [ ]:
# Paso 1: Dividir en chunks
df_posts_chunked_1000 = add_chunks_column(posts_train_mono_sample_1000, 'ocr_text', max_tokens=512)

# Paso 2: Generar embeddings por chunks
posts_train_mono_sample_1000 = generate_chunk_embeddings_batched(df_posts_chunked_1000, small_model, new_column_name='ocr_text_embedding',device="mps")

posts_train_mono_sample_1000['ocr_text_embedding']=posts_train_mono_sample_1000['ocr_text_eng_embedding']
posts_train_mono_sample_1000.drop(columns='ocr_text_eng_embedding', inplace=True)

In [ ]:
embedding_columns_posts = ['ocr_text_embedding', 'ocr_text_eng_embedding']
embedding_columns_fact_checks = ['claim_title_embedding', 'claim_title_eng_embedding']

In [185]:
prepare_df_for_saving(posts_train_mono_sample_1000, embedding_columns_posts).to_csv('data/posts_train_mono_sample_1000.csv', index=False)

In [186]:
prepare_df_for_saving(fact_checks_mono_sample_1000, embedding_columns_fact_checks).to_csv('data/fact_checks_mono_sample_1000.csv', index=False)

In [ ]:
import ast
import numpy as np
import pandas as pd

import numpy as np
import ast

def restore_embeddings(df, embedding_columns):
    for col in embedding_columns:
        def safe_eval(x):
            try:
                if isinstance(x, np.ndarray):  # Ya es un array de numpy
                    return x.astype('float32')
                elif isinstance(x, list):  # Es una lista estándar
                    return np.array(x, dtype='float32')
                elif isinstance(x, str):  # Es una cadena que necesita evaluación
                    return np.array(ast.literal_eval(x), dtype='float32')
                else:
                    return np.nan  # Cualquier otro tipo es inválido
            except Exception as e:
                print(f"Error evaluando fila: {x}, Error: {e}")
                return np.nan
        
        # Aplicar conversión segura
        df[col] = df[col].apply(safe_eval)
    return df

## Retrieval

In [29]:
import numpy as np
import pandas as pd
import faiss

def find_top_k_similar_faiss(df_1, df_2, chunk_col='chunk_embeddings', embedding_col='embedding', k=10):
    """
    Encuentra los k elementos más similares en df_2 para cada entrada en df_1 usando FAISS.
    Devuelve un DataFrame con los post_id y fact_check_id correspondientes en lugar de índices.

    Args:
        df_1 (DataFrame): DataFrame que contiene los embeddings de los posts y la columna 'post_id'.
        df_2 (DataFrame): DataFrame que contiene los embeddings de los fact checks y la columna 'fact_check_id'.
        chunk_col (str): Columna en df_1 que contiene los embeddings (listas de arrays).
        embedding_col (str): Columna en df_2 que contiene los embeddings.
        k (int): Número de elementos más similares a devolver.

    Returns:
        DataFrame: DataFrame con columnas 'post_id' y 'top_k' (lista de fact_check_id y similitudes).
    """
    # Indexar los embeddings de df_2
    dimension = len(df_2[embedding_col].iloc[0])
    index = faiss.IndexFlatL2(dimension)
    
    # Convertir embeddings de df_2 a float32
    embeddings_2 = np.vstack(df_2[embedding_col]).astype('float32')
    faiss.normalize_L2(embeddings_2)
    index.add(embeddings_2)

    # Inicializar lista de resultados
    results = []

    for _, row_1 in df_1.iterrows():
        all_matches = []

        for chunk in row_1[chunk_col]:
            # Convertir chunk a float32
            chunk = np.array(chunk, dtype='float32').reshape(1, -1)
            faiss.normalize_L2(chunk)
            distances, indices = index.search(chunk, k)

            for dist, idx in zip(distances[0], indices[0]):
                # Obtener el fact_check_id correspondiente al índice
                fact_check_id = df_2.iloc[idx]['fact_check_id']
                all_matches.append({"fact_check_id": fact_check_id, "similarity": 1 - dist})

        # Ordenar por similitud y devolver top_k
        all_matches = sorted(all_matches, key=lambda x: x['similarity'], reverse=True)[:k]

        # Añadir resultado con post_id
        results.append({"post_id": row_1['post_id'], "top_k": all_matches})

    return pd.DataFrame(results)

In [74]:
def success_at_10(predictions_df, pairs_df):
    """
    Calcula la métrica Success-at-10.

    Args:
        predictions_df (DataFrame): Contiene las predicciones con columnas 'post_id' y 'predicted_fact_check_ids'.
        pairs_df (DataFrame): Contiene las respuestas correctas con columnas 'post_id' y 'fact_check_id'.

    Returns:
        DataFrame: DataFrame con una nueva columna 'success_at_10' indicando si la predicción fue correcta.
    """
    # Crear un diccionario de respuestas correctas: {post_id: set(fact_check_ids)}
    correct_answers = pairs_df.groupby('post_id')['fact_check_id'].apply(set).to_dict()

    # Evaluar cada post_id en las predicciones
    success_scores = []

    for _, row in predictions_df.iterrows():
        post_id = row['post_id']
        predicted_ids = set(int(item['fact_check_id']) for item in row['top_k'])

        # Verificar si alguna predicción es correcta
        success = 1 if post_id in correct_answers and predicted_ids & correct_answers[post_id] else 0
        success_scores.append(success)
    
    # Añadir la columna 'success_at_10' al DataFrame
    predictions_df['success_at_10'] = success_scores

    return predictions_df

In [ ]:
results_ocr_text_orig_100=find_top_k_similar_faiss(df_1=posts_train_mono_sample, df_2=fact_checks_mono_sample_100, chunk_col='ocr_text_embedding', embedding_col='claim_title_embedding', k=10)

results_ocr_text_eng_100=find_top_k_similar_faiss(df_1=posts_train_mono_sample, df_2=fact_checks_mono_sample_100, chunk_col='ocr_text_eng_embedding', embedding_col='claim_title_eng_embedding', k=10)

In [ ]:
results_ocr_text_orig_1000=find_top_k_similar_faiss(df_1=posts_train_mono_sample_1000, df_2=fact_checks_mono_sample_1000, chunk_col='ocr_text_embedding', embedding_col='claim_title_embedding', k=10)

In [107]:
results_ocr_text_orig_leng=success_at_10(results_ocr_text_orig_1000, df_fact_check_post_mapping)

In [81]:
results_ocr_text_orig_leng['success_at_10'].mean()

np.float64(0.95)

In [116]:
df_fact_check_post_mapping.groupby('post_id')['fact_check_id'].count().mean()

np.float64(1.1707749681644533)

In [117]:
results_ocr_text_orig_1000['success_at_10'].mean()

np.float64(0.844)

## English embeddings sample 1000

In [124]:
# Paso 1: Dividir en chunks
df_posts_chunked_1000_eng = add_chunks_column(posts_train_mono_sample_1000, 'ocr_text_eng', max_tokens=512)

# Paso 2: Generar embeddings por chunks
posts_train_mono_sample_1000 = generate_chunk_embeddings_batched(df_posts_chunked_1000_eng, small_model, new_column_name='ocr_text_eng_embedding',device="mps")


In [125]:

results_ocr_text_eng_1000=find_top_k_similar_faiss(df_1=posts_train_mono_sample_1000, df_2=fact_checks_mono_sample_1000, chunk_col='ocr_text_eng_embedding', embedding_col='claim_title_eng_embedding', k=10)

results_ocr_text_eng_1000=success_at_10(results_ocr_text_eng_1000, df_fact_check_post_mapping)

results_ocr_text_eng_1000['success_at_10'].mean()

np.float64(0.758)